# CHIMERA Model Comparison

This notebook demonstrates how to compare multiple AI models using the CHIMERA benchmark framework.

## Contents

1. Setup and Configuration
2. Running Evaluations
3. Computing Rankings
4. Visualizing Comparisons
5. Statistical Analysis

## 1. Setup and Configuration

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# CHIMERA imports
from chimera.evaluation import (
    EvaluationPipeline,
    PipelineConfig,
    ModelComparison,
)

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Setup complete!")

In [ ]:
# Define models to compare
MODELS = [
    ("gemini", "gemini-2.0-flash"),
    ("gemini", "gemini-2.0-pro"),
    ("openai", "gpt-4o-mini"),
    ("openai", "gpt-4o"),
]

# Evaluation tracks
TRACKS = [
    "calibration",
    "error_detection",
    "knowledge_boundary",
    "self_correction",
]

# Configuration
N_TASKS = 100
SEED = 42

print(f"Models: {[m[1] for m in MODELS]}")
print(f"Tracks: {TRACKS}")
print(f"Tasks per track: {N_TASKS}")

## 2. Running Evaluations

We'll run each model on all tracks and collect the results.

In [ ]:
# NOTE: Running actual evaluations requires API credentials
# Uncomment the code below to run real evaluations

'''
comparison = ModelComparison()

for provider, model_name in MODELS:
    print(f"\nEvaluating {model_name}...")
    
    config = PipelineConfig(
        tracks=TRACKS,
        model_provider=provider,
        model_name=model_name,
        n_tasks=N_TASKS,
        seed=SEED,
        output_dir=f"results/{model_name}",
    )
    
    pipeline = EvaluationPipeline(config)
    results = pipeline.run()
    
    comparison.add_model_results(model_name, results)
    print(f"  Overall: {results.overall_score:.2%}")
'''

print("Evaluation code shown above (uncomment to run with real API calls)")

In [ ]:
# For demonstration, we'll use synthetic data
np.random.seed(42)

# Synthetic results (realistic-looking scores)
synthetic_results = {
    "gemini-2.0-flash": {
        "calibration": 0.78,
        "error_detection": 0.72,
        "knowledge_boundary": 0.75,
        "self_correction": 0.68,
    },
    "gemini-2.0-pro": {
        "calibration": 0.85,
        "error_detection": 0.81,
        "knowledge_boundary": 0.83,
        "self_correction": 0.79,
    },
    "gpt-4o-mini": {
        "calibration": 0.74,
        "error_detection": 0.78,
        "knowledge_boundary": 0.71,
        "self_correction": 0.73,
    },
    "gpt-4o": {
        "calibration": 0.82,
        "error_detection": 0.84,
        "knowledge_boundary": 0.80,
        "self_correction": 0.82,
    },
}

# Compute overall scores (weighted average)
for model, tracks in synthetic_results.items():
    overall = sum(tracks.values()) / len(tracks)
    synthetic_results[model]["overall"] = overall

# Create DataFrame for easy analysis
df = pd.DataFrame(synthetic_results).T
df = df.round(3)
print("\nSynthetic Benchmark Results:")
display(df)

## 3. Computing Rankings

In [ ]:
# Rank models by overall score
rankings = df.sort_values('overall', ascending=False)
rankings['rank'] = range(1, len(rankings) + 1)

print("Model Rankings (Overall Score)")
print("="*50)
for idx, row in rankings.iterrows():
    print(f"{int(row['rank'])}. {idx}: {row['overall']:.2%}")

In [ ]:
# Rank by each track
print("\nBest Model per Track")
print("="*50)

for track in TRACKS:
    best_model = df[track].idxmax()
    best_score = df[track].max()
    print(f"{track:20s}: {best_model} ({best_score:.2%})")

In [ ]:
# Compute performance deltas
print("\nPerformance Deltas (vs best model)")
print("="*50)

best_overall = rankings.iloc[0]
best_model = rankings.index[0]

print(f"Best model: {best_model} ({best_overall['overall']:.2%})\n")

for model in rankings.index[1:]:
    delta = df.loc[model, 'overall'] - best_overall['overall']
    print(f"{model}: {delta:+.2%} from best")

## 4. Visualizing Comparisons

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(12, 6))

models = list(synthetic_results.keys())
x = np.arange(len(models))
width = 0.18

colors = ['#4C78A8', '#F58518', '#E45756', '#72B7B2']

for i, track in enumerate(TRACKS):
    scores = [synthetic_results[m][track] for m in models]
    ax.bar(x + i * width, scores, width, label=track.replace('_', ' ').title(), 
           color=colors[i], edgecolor='black', linewidth=0.5)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance by Track', fontsize=14)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(models, rotation=15)
ax.legend(loc='lower right')
ax.set_ylim(0, 1)
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, label='80% threshold')

plt.tight_layout()
plt.show()

In [ ]:
# Radar chart for multi-dimensional comparison
def create_radar_chart(models_data, tracks, title="Model Comparison"):
    """Create a radar chart comparing models across tracks."""
    n_tracks = len(tracks)
    angles = np.linspace(0, 2 * np.pi, n_tracks, endpoint=False).tolist()
    angles += angles[:1]  # Complete the circle
    
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
    
    colors = ['#4C78A8', '#F58518', '#E45756', '#72B7B2']
    
    for idx, (model, scores_dict) in enumerate(models_data.items()):
        scores = [scores_dict[t] for t in tracks]
        scores += scores[:1]  # Complete the circle
        
        ax.plot(angles, scores, 'o-', linewidth=2, color=colors[idx], 
                label=model, markersize=6)
        ax.fill(angles, scores, alpha=0.15, color=colors[idx])
    
    # Customize the chart
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([t.replace('_', '\n').title() for t in tracks], fontsize=11)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['20%', '40%', '60%', '80%', '100%'], fontsize=9)
    ax.set_title(title, fontsize=14, pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
    
    return fig

fig = create_radar_chart(synthetic_results, TRACKS, "CHIMERA Benchmark Comparison")
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of all scores
fig, ax = plt.subplots(figsize=(10, 6))

# Prepare data
heatmap_data = df[TRACKS + ['overall']].values
models = df.index.tolist()
columns = TRACKS + ['overall']

# Create heatmap
im = ax.imshow(heatmap_data, cmap='RdYlGn', aspect='auto', vmin=0.6, vmax=0.9)

# Labels
ax.set_xticks(range(len(columns)))
ax.set_xticklabels([c.replace('_', ' ').title() for c in columns], rotation=45, ha='right')
ax.set_yticks(range(len(models)))
ax.set_yticklabels(models)

# Add score annotations
for i in range(len(models)):
    for j in range(len(columns)):
        score = heatmap_data[i, j]
        text_color = 'white' if score < 0.75 else 'black'
        ax.text(j, i, f'{score:.1%}', ha='center', va='center', 
                color=text_color, fontsize=10, fontweight='bold')

ax.set_title('Model Performance Heatmap', fontsize=14)
plt.colorbar(im, label='Score')
plt.tight_layout()
plt.show()

## 5. Statistical Analysis

In [ ]:
# Simulate per-task scores for statistical analysis
np.random.seed(42)

# Generate per-task scores (beta distribution around mean)
per_task_scores = {}
for model, tracks in synthetic_results.items():
    per_task_scores[model] = {}
    for track in TRACKS:
        mean_score = tracks[track]
        # Generate 100 task scores with some variance
        a = mean_score * 10
        b = (1 - mean_score) * 10
        scores = np.random.beta(a, b, N_TASKS)
        per_task_scores[model][track] = scores

print(f"Generated {N_TASKS} task scores per track per model")

In [ ]:
# Compute confidence intervals
from scipy import stats

print("95% Confidence Intervals for Overall Scores")
print("="*60)

for model in per_task_scores:
    # Combine all track scores
    all_scores = np.concatenate([per_task_scores[model][t] for t in TRACKS])
    
    mean = all_scores.mean()
    sem = stats.sem(all_scores)
    ci = stats.t.interval(0.95, len(all_scores)-1, loc=mean, scale=sem)
    
    print(f"{model:20s}: {mean:.2%} [{ci[0]:.2%}, {ci[1]:.2%}]")

In [ ]:
# Pairwise statistical comparisons
print("\nPairwise Comparisons (paired t-test)")
print("="*60)

models_list = list(per_task_scores.keys())

for i in range(len(models_list)):
    for j in range(i + 1, len(models_list)):
        model_a = models_list[i]
        model_b = models_list[j]
        
        # Get overall scores for each model
        scores_a = np.concatenate([per_task_scores[model_a][t] for t in TRACKS])
        scores_b = np.concatenate([per_task_scores[model_b][t] for t in TRACKS])
        
        # Paired t-test
        t_stat, p_value = stats.ttest_rel(scores_a, scores_b)
        
        diff = scores_a.mean() - scores_b.mean()
        sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else ""
        
        print(f"{model_a} vs {model_b}:")
        print(f"  Difference: {diff:+.2%}")
        print(f"  p-value: {p_value:.4f} {sig}")
        print()

In [ ]:
# Effect size (Cohen's d)
def cohens_d(group1, group2):
    """Compute Cohen's d effect size."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(), group2.var()
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (group1.mean() - group2.mean()) / pooled_std

print("Effect Sizes (Cohen's d)")
print("="*60)
print("Interpretation: |d| < 0.2 = negligible, 0.2-0.5 = small, 0.5-0.8 = medium, > 0.8 = large\n")

# Compare best vs worst
best_model = rankings.index[0]
worst_model = rankings.index[-1]

scores_best = np.concatenate([per_task_scores[best_model][t] for t in TRACKS])
scores_worst = np.concatenate([per_task_scores[worst_model][t] for t in TRACKS])

d = cohens_d(scores_best, scores_worst)

interpretation = "negligible" if abs(d) < 0.2 else "small" if abs(d) < 0.5 else "medium" if abs(d) < 0.8 else "large"

print(f"{best_model} vs {worst_model}:")
print(f"  Cohen's d: {d:.3f} ({interpretation} effect)")

In [ ]:
# Visualize confidence intervals
fig, ax = plt.subplots(figsize=(10, 6))

models_sorted = rankings.index.tolist()
y_pos = range(len(models_sorted))

means = []
ci_lows = []
ci_highs = []

for model in models_sorted:
    all_scores = np.concatenate([per_task_scores[model][t] for t in TRACKS])
    mean = all_scores.mean()
    sem = stats.sem(all_scores)
    ci = stats.t.interval(0.95, len(all_scores)-1, loc=mean, scale=sem)
    
    means.append(mean)
    ci_lows.append(ci[0])
    ci_highs.append(ci[1])

# Plot
colors = ['#4C78A8', '#F58518', '#E45756', '#72B7B2']
ax.barh(y_pos, means, xerr=[np.array(means) - np.array(ci_lows), 
                            np.array(ci_highs) - np.array(means)],
        color=colors, edgecolor='black', capsize=5, alpha=0.8)

ax.set_yticks(y_pos)
ax.set_yticklabels(models_sorted)
ax.set_xlabel('Overall Score', fontsize=12)
ax.set_title('Model Performance with 95% Confidence Intervals', fontsize=14)
ax.set_xlim(0.6, 0.9)
ax.axvline(x=0.8, color='gray', linestyle='--', alpha=0.5)

# Add score annotations
for i, (mean, ci_h) in enumerate(zip(means, ci_highs)):
    ax.text(ci_h + 0.01, i, f'{mean:.1%}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## Conclusion

This notebook demonstrated:

1. **Running evaluations** on multiple models using CHIMERA
2. **Computing rankings** by overall and per-track performance
3. **Visualizing comparisons** with bar charts, radar charts, and heatmaps
4. **Statistical analysis** including confidence intervals, t-tests, and effect sizes

### Key Findings (from synthetic data)

- **Best overall**: gemini-2.0-pro
- **Best per-track**: Different models excel at different tracks
- **Statistical significance**: Larger models show significant improvements

### Next Steps

- Run with real API calls to get actual results
- Increase N_TASKS for more statistical power
- Compare across more models and tracks